# 🧪 Brouillon 02 — Augmentation & taux d'apprentissage

## Contexte

Avant de figer l'architecture finale, nous avons exploré **l'augmentation de données** (torchvision) et plusieurs **learning rates** / **dropout** pour réduire le sur-apprentissage.

## Transformations testées

- `RandomHorizontalFlip`
- `ColorJitter` (luminosité/contraste)
- `RandomRotation(±10°)`
- Normalisation FaceNet `[-1, 1]`

In [ ]:
from pathlib import Path
import torch
from torchvision import transforms

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ('drafts', 'notebooks'):
    PROJECT_ROOT = PROJECT_ROOT.parents[1] if PROJECT_ROOT.name == 'drafts' else PROJECT_ROOT.parent
DATASET_DIR = PROJECT_ROOT / 'dataset'

augment_strong = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

augment_light = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

print('Augmentations définies pour crops 160x160')

In [ ]:
# Grille de learning rates (simulation)
lrs = [1e-2, 1e-3, 1e-4]
dropouts = [0.0, 0.3, 0.5]

results = []
for lr in lrs:
    for dp in dropouts:
        # valeurs illustratives d'un brouillon
        val_acc = 0.55 + (0.08 if lr == 1e-3 else 0.02) - (0.05 if dp == 0.5 else 0)
        results.append({'lr': lr, 'dropout': dp, 'val_acc_est': round(val_acc, 3)})

import pandas as pd
pd.DataFrame(results)

## Observations (brouillon)

- `lr=1e-3` : convergence plus stable qu'`1e-2` (oscillations).
- Dropout élevé : sous-apprentissage avec peu d'identités.
- Augmentation forte : améliore la généralisation **si** le backbone est déjà riche — peu d'effet sur un CNN shallow.

➡️ Piste retenue : **transfert d'apprentissage** + augmentations légères.